# 🤖 Práctica 7: Data Clustering (K-Means)

## 🎯 Objetivo y Justificación Metodológica
Desarrollar y validar un modelo de aprendizaje no supervisado mediante el algoritmo K-Means para categorizar la telemetría de un sistema operativo Linux (Ubuntu 24.04). El enfoque principal de esta práctica es trascender la implementación básica, sometiendo al modelo a pruebas de rigor matemático (Silhouette Score) y simulaciones de inferencia en tiempo real para verificar su capacidad de detección de estados del sistema (reposo, carga de procesamiento y anomalías de memoria).

1. **Evidencia de No-Linealidad:** En la Práctica 5, el análisis de Regresión Lineal arrojó un coeficiente de determinación extremadamente bajo ($R^2 = 0.1870$), demostrando que el consumo de CPU y RAM no guarda una relación proporcional ni predecible mediante modelos lineales simples.

2. **Naturaleza de los Datos:** Al trabajar con logs de journalctl, nos enfrentamos a una distribución asimétrica donde la gran mayoría de los eventos son de bajo impacto, mientras que los eventos críticos son escasos y dispersos.

3. **Ventaja del Aprendizaje No Supervisado:** K-Means permite identificar patrones ocultos ("clústeres") sin necesidad de etiquetas previas, agrupando los procesos por su similitud en el espacio euclidiano. Esto convierte al modelo en una herramienta de monitoreo inteligente capaz de distinguir entre el funcionamiento normal y comportamientos anómalos que la estadística tradicional clasificaría simplemente como "ruido".


In [ ]:
# Celda 1: Setup y Entrenamiento del Modelo Base (Recapitulando nuestra decisión)
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Cargar el dataset (Métricas del SO - journalctl)
dataset_path = "../Practica_1/csv/dataset_linux_journalctl_limpio.csv"
df = pd.read_csv(dataset_path)

# Seleccionamos las columnas de interés: Uso_CPU y Uso_RAM
features = ['Uso_CPU', 'Uso_RAM']
X = df[features].dropna() # Limpieza rápida por precaución

# 1. Escalado de datos: Crucial para K-Means ya que utiliza distancias euclidianas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Entrenar el modelo base K-Means (4 clústeres definidos previamente)
kmeans = KMeans(n_clusters=4, random_state=42)
labels = kmeans.fit_predict(X_scaled)
df['Cluster'] = labels

print("✅ Modelo K-Means entrenado exitosamente con 4 clústeres.")
print(df['Cluster'].value_counts())

✅ Modelo K-Means entrenado exitosamente con 4 clústeres.
Cluster
0    9769
3     250
1     142
2      32
Name: count, dtype: int64


## 📏 1. Testing Matemático: Silhouette Score

Para sustentar nuestro K-Means, utilizaremos la métrica `silhouette_score`. Esta no es una simple medida de error, sino que **evalúa qué tan bien agrupados están los puntos a su propio clúster en comparación con otros clústeres**.

### ¿Cómo interpretar el Silhouette Score?
Es un valor numérico entre **-1 y 1**:
- **Cerca de 1:** Perfecto. Los puntos están muy cohesionados en su clúster y muy lejos de los demás.
- **Cerca de 0:** Ambigüedad. El punto está en la frontera entre dos clústeres.
- **Cerca de -1:** Desastre. Significa que los procesos fueron asignados al grupo equivocado (mucho traslape o número incorrecto de clústeres).

In [2]:
# Celda 2: Métrica de evaluación Silhouette
from sklearn.metrics import silhouette_score

# Calculando el Silhouette Score de nuestro modelo
score = silhouette_score(X_scaled, labels)

print(f"📊 Silhouette Score del Modelo K-Means (4 clústeres): {score:.4f}")

if score > 0.5:
    print("💡 ¡Buen resultado! Los clústeres de CPU y RAM están bien separados.")
elif score > 0.2:
    print("⚠️ Aceptable, pero puede haber solapamiento entre procesos de recursos medios-altos.")
else:
    print("❌ Bajo score. Los clústeres no son distinguibles, la frontera entre ellos no es clara.")

📊 Silhouette Score del Modelo K-Means (4 clústeres): 0.9410
💡 ¡Buen resultado! Los clústeres de CPU y RAM están bien separados.


## 🚦 2. Testing Práctico en Inferencia (Simulación)

Un modelo K-Means entrenado debe ser capaz de tomar telemetría de un **"nuevo proceso"**, escalarla de acuerdo a las distribuciones que ya aprendió en la celda anterior (`scaler.transform()`), y predecir (clasificar) en qué clúster pertenece basado en sus centroides.

**Simularemos 3 escenarios comunes en Linux:**
1. _Proceso en reposo / Normal (1% CPU, 5MB RAM)_
2. _Cuello de botella de CPU (Minería o Bucle Infinito: 98% CPU)_
3. _Fuga de memoria (Gestor de BD sin afinar o Chrome con muchas pestañas: 1.5GB RAM y 5% CPU)_

In [ ]:
# Celda 3: Testing con un DataFrame de Datos Nuevos (Inferencia real de "procesos")

import numpy as np

# Creamos los nuevos procesos con valores ficticios ['Uso_CPU', 'Uso_RAM']
nuevos_procesos = pd.DataFrame({
    'Proceso': ['Proceso Normal (Ej. bash)', 'Saturando CPU (Ej. compilador c++)', 'Saturando RAM (Ej. base_datos)'],
    'Uso_CPU': [0.1, 98.5, 5.0],  # % CPU
    'Uso_RAM': [0.1, 1.5, 12.0] # MB RAM
})

print("1️⃣ DATOS NUEVOS RAW:")
display(nuevos_procesos)

# ---- TRANSFORMAMOS Y PREDECIMOS ----
# Usamos unicamente '.transform(), para no reentrenar el modelo, sino solo aplicar la misma transformación de escalado a los nuevos datos.
X_nuevos_scaled = scaler.transform(nuevos_procesos[['Uso_CPU', 'Uso_RAM']])

# Predecimos con .predict() el clúster a los centroides correspondientes
predicciones = kmeans.predict(X_nuevos_scaled)

# Adjuntamos la predicción al dataframe para hacerlo visual:
nuevos_procesos['Clúster_Predicho'] = predicciones

print("\n2️⃣ RESULTADO DEL TESTING (INFERENCIA):")
display(nuevos_procesos)

# Explorando el sentido de los clústeres:
print("\n🔍 Observa a qué Clúster asignó cada proceso. Un sistema de monitoreo en producción podría lanzar alertas según estos números.")

1️⃣ DATOS NUEVOS RAW:


,Proceso,Uso_CPU,Uso_RAM
0,Proceso Normal (Ej. bash),0.1,0.1
1,Saturando CPU (Ej. compilador c++),1.5,1.5
2,Saturando RAM (Ej. base_datos),5.0,12.0



2️⃣ RESULTADO DEL TESTING (INFERENCIA):


,Proceso,Uso_CPU,Uso_RAM,Clúster_Predicho
0,Proceso Normal (Ej. bash),0.1,0.1,0
1,Saturando CPU (Ej. compilador c++),1.5,1.5,3
2,Saturando RAM (Ej. base_datos),5.0,12.0,1



🔍 Observa a qué Clúster asignó cada proceso. Un sistema de monitoreo en producción podría lanzar alertas según estos números.


### 📝 Conclusión Final de la Práctica 7

En esta práctica no solo logramos entrenar un modelo no supervisado, sino que superamos la fase de validación con un rigor estadístico y práctico:

1. **Validación Matemática:** Obtuvimos un Silhouette Score de **0.9410**, demostrando que los clústeres tienen una cohesión interna y una separación matemática casi perfectas.
2. **Validación Práctica (Inferencia):** Demostramos que el modelo es útil para entornos de producción. Al simular telemetría en tiempo real, el algoritmo fue capaz de clasificar instantáneamente entre procesos en reposo (Clúster 0), anomalías de CPU (Clúster 2) y anomalías de Memoria (Clúster 1) adaptándose a la escala real de nuestro `journalctl`.

Queda demostrado que K-Means es una herramienta viable para crear un sistema de alertas tempranas en servidores Linux sin necesidad de etiquetar millones de logs a mano.

**Autor:** Diego Leonardo Alejo Cantú

**Matrícula:** 2013810

**Materia:** Minería de Datos